[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skous2/is455-w2027/blob/main/module_03/Worksheet_3A_Business_Understanding.ipynb)

# Worksheet 3A · Business understanding

**Optional and not graded.** Work through it before Tuesday's class (Jan 26), where we walk through these same problems. Expect about 30–40 minutes.

**Before you start,** choose **File → Save a copy in Drive** so your work is saved. Try each problem yourself first. When you get stuck, paste the tutor prompt below into your AI assistant so it gives you hints instead of the answer. Most answers in this worksheet are written explanations, so write them in your own words. Interview 3 on Friday asks you to explain these ideas out loud.

Each section starts with a short worked example from a made-up smoothie shop, then gives you the same kind of problem for Canyon Fitness. Each section also links the reading that covers it. The readings are short web pages, and you can read each one when you reach its section.

## When you get stuck

Copy the whole prompt below into your AI assistant and fill in the two parts in brackets. It tells the AI to explain the idea and give one hint at a time instead of writing the code for you. After you solve the problem, it gives you one new practice problem to try on your own. If your AI tool can edit this notebook directly, tell it not to, so you type the code yourself.

```text
I'm a student in IS 455, a machine learning class, learning Python and pandas. Do not write the solution for me.
Here is the problem and my code so far. Explain the idea I'm missing in plain language, give me one hint at a time,
and wait for me to try again after each hint.
When I have solved it, give me ONE new practice problem that tests the same idea with different data and a small twist.
Don't help with the practice problem unless I ask.

PROBLEM:
[paste the problem text]

MY CODE AND THE ERROR OR RESULT:
[paste your code and what happened]
```

## The request

You are an analyst at **Canyon Fitness**. This email arrives from the VP of Membership.

> *"We're losing too many members. Can you build something that predicts who's going to churn so we can stop them?"*

That is the whole request. This module covers **business understanding**, the first phase of CRISP-DM, where you turn a vague request like this into a prediction problem worth solving.

## 1. Start with the business question

*Read [CRISP-DM 1.0 guide, p. 13](https://public.dhe.ibm.com/software/analytics/spss/documentation/modeler/14.2/es/CRISP-DM.pdf) for the six phases, and [Google, "Understand the problem"](https://developers.google.com/machine-learning/problem-framing/problem) for why the goal comes before the model.*

**Worked example.** Suppose a smoothie-shop owner asks you to "use AI to sell more smoothies." Good first questions are ones whose answers would change what you build. *What would you do with a prediction, send a coupon or change the menu?* decides what the model predicts. *Which customers matter most, new ones or regulars?* decides who the model scores. *How will you know it worked?* sets the success measure before you build anything.

**Your turn.**

**1a.** CRISP-DM has six phases. Name them in order, and explain in one sentence why business understanding comes first.

*Your answer here.*

**1b.** Write three questions you would ask the VP of Membership before building anything. Choose questions whose answers would change what you build, and say what each answer would change.

*Your answer here.*

## 2. The prediction point

*[Google, "Understand the problem"](https://developers.google.com/machine-learning/problem-framing/problem) explains why a feature must be available when the prediction is made.*

**Worked example.** The **prediction point** is the moment the prediction is made. Anything recorded after that moment can't be a feature, because it doesn't exist yet when the model runs. A column like `receipt_total_this_week` would make a model look accurate in testing, because it partly contains the answer, but on Monday morning it is empty. That problem is called **leakage**. `pd.crosstab` counts how two columns line up, and here it shows the receipt total gives the answer away.

In [ ]:
import pandas as pd

# Example: on Monday morning, the shop predicts who will buy a pastry this week
shop = pd.DataFrame({
    "customer":                ["Ana", "Ben", "Cy", "Dee"],
    "visits_last_week":        [5, 1, 3, 4],              # known on Monday
    "member":                  [True, False, True, False], # known on Monday
    "pastry_this_week":        [True, False, False, True], # the answer, known on Sunday
    "receipt_total_this_week": [31.50, 4.25, 6.00, 22.75], # also not known until Sunday
})
display(shop)
pd.crosstab(shop["pastry_this_week"], shop["receipt_total_this_week"] > 10)

**Your turn.**

The data is a snapshot of every active member on **October 1, 2026**, the prediction point, plus what happened to each member over the next 92 days. Run the cell below. `parse_dates` tells pandas to read those columns as dates instead of text.

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/canyon_fitness_members.csv"
members = pd.read_csv(DATA_URL, parse_dates=["join_date", "next_visit_date", "cancel_date"])
snapshot = pd.Timestamp("2026-10-01")

print(members.shape)
members.head()

**2a.** What does one row represent?

*Your answer here.*

**2b.** Which columns would you **not** know on October 1? For each one, explain why using it in a model would be a problem.

*Your answer here.*

**2c.** Write code with `pd.crosstab` that shows how `cancel_date` being filled in lines up with `cancellation_reason` being filled in. Use `.notna()` on each column. What does the result show?

In [ ]:
# Your code here


*Your answer here.*

## 3. Define the target

*[Google, "Framing an ML problem"](https://developers.google.com/machine-learning/problem-framing/ml-framing) covers the model's goal and *proxy labels*, which are measurable stand-ins for an outcome you can't measure directly. Each churn definition below is a proxy label.*

**Worked example.** The same customers can have very different "lapse" rates depending on how you define a lapse. `monday + pd.Timedelta(days=7)` is the date one week after Monday. A blank date (`NaT`) is never greater than another date, so a definition that counts blanks needs `.isna()` joined with `|`.

In [ ]:
monday = pd.Timestamp("2026-09-28")
visits = pd.DataFrame({
    "customer":   ["Ana", "Ben", "Cy", "Dee"],
    "next_visit": pd.to_datetime(["2026-09-29", None, "2026-10-20", "2026-10-02"]),
})

lapsed_in_week = visits["next_visit"].isna() | (visits["next_visit"] > monday + pd.Timedelta(days=7))
never_back = visits["next_visit"].isna()
print(lapsed_in_week.mean(), never_back.mean())   # 0.5 and 0.25

**Your turn.**

Here are three reasonable definitions of churn for Canyon Fitness.

| Definition | A member has churned if they |
| --- | --- |
| A | have no visit in the 30 days after October 1 |
| B | have no visit at all by January 1 |
| C | cancelled by January 1 |

**3a.** Compute the churn rate under each definition and store them as `churn_A`, `churn_B` and `churn_C`.

In [ ]:
# Your code here


**3b.** The rates are very different. Explain in one or two sentences why the definition matters so much for what the model will learn.

*Your answer here.*

**3c.** The retention team sends offers **once a month**, and an offer only helps if it reaches a member *before* they decide to leave. Which definition fits that plan best, and why?

*Your answer here.*

## 4. What do mistakes cost?

*[Google, "Thresholds and the confusion matrix"](https://developers.google.com/machine-learning/crash-course/classification/thresholding) defines false positives and false negatives and explains why the cutoff depends on which mistake costs more.*

**Worked example.** Suppose the smoothie shop mails a \$1 coupon, a lost regular is worth \$50, and the coupon brings back 10% of customers who were about to leave. Sending a coupon to someone who would have stayed (a false positive) wastes \$1. Missing someone who leaves (a false negative) gives up the chance to save them, worth 10% of \$50, or \$5 on average. A coupon pays for itself when the customer's chance of leaving is above \$1 ÷ (0.10 × \$50) = 20%.

In [ ]:
coupon_cost = 1
customer_value = 50
save_rate = 0.10

break_even = coupon_cost / (save_rate * customer_value)
print(f"Send a coupon when the chance of leaving is above {break_even:.0%}")

**Your turn.**

Assume these numbers for Canyon Fitness.

- A retention offer, a free month of personal training, costs Canyon Fitness **\$20** per member who receives it.
- A member who leaves is worth about **\$360** in lost revenue, about \$45 a month for 8 months.
- The offer convinces about **25%** of members who were going to leave to stay.

**4a.** Which mistake is worse, sending an offer to someone who would have stayed (a *false positive*) or missing someone who leaves (a *false negative*)? Use the numbers.

*Your answer here.*

**4b.** Compute the break-even chance of churning, above which an offer is worth sending. Store it as `break_even`, then explain the result in plain words.

In [ ]:
# Your code here


*Your answer here.*

## 5. Do we even need a model?

*The section "Clear use case for ML" in [Google, "Understand the problem"](https://developers.google.com/machine-learning/problem-framing/problem) recommends comparing any model with a simple rule first.*

**Worked example.** A simple rule can be a strong baseline. Compare the outcome rate for customers the rule flags with everyone else, then check what share of all the customers you care about the rule actually catches. A rule can be accurate for the people it flags and still miss most of the problem.

In [ ]:
toy = pd.DataFrame({
    "days_since_visit": [2, 30, 5, 45, 1, 25, 3, 8],
    "lapsed":           [False, True, False, False, True, True, False, True],
})
rule = toy["days_since_visit"] > 21

display(toy.groupby(rule)["lapsed"].agg(["count", "mean"]))   # lapse rate with and without the rule
print(toy.loc[rule, "lapsed"].sum() / toy["lapsed"].sum())     # share of all lapsed customers the rule catches

**Your turn.**

**5a.** Consider a simple rule that sends the offer to every member with more than 21 days since their last visit. Using definition B, compare the churn rate for members the rule flags with everyone else. Then compute the share of all churners the rule catches.

In [ ]:
# Your code here


**5b.** Compare both churn rates with your break-even from 4b. Is the rule worth using? What would a model have to do better to be worth building instead?

*Your answer here.*

## 6. Prediction is not causation, and some features need a second look

*[Google, "Fairness: Types of bias"](https://developers.google.com/machine-learning/crash-course/fairness/types-of-bias) describes how a model can treat groups of people unfairly even when no one intends it to.*

**Worked example.** Customers who use the smoothie shop's app spend more. That doesn't mean making everyone install the app would raise spending, because app users may already be the shop's biggest fans. A model can use app use to *find* big spenders even though the app doesn't *cause* the spending. Some features also need care for a different reason. A model that uses a customer's neighborhood to hand out coupons could end up giving most of the discounts to one part of town.

**Your turn.**

**6a.** Write code that compares the definition-B churn rate for members on autopay with those who aren't. Then answer this. Should Canyon Fitness require every member to use autopay to reduce churn? Explain the difference between prediction and causation in your answer.

In [ ]:
# Your code here


*Your answer here.*

**6b.** Age helps predict churn. Should the model use age when deciding who gets a retention offer? Give one reason it might be fine and one thing that could go wrong.

*Your answer here.*

## Bring to class

Write down anything that confused you. Tuesday's walk-through goes through every problem in this worksheet, and the answer key is posted after class.

*Your answer here.*